# Field Shift — rotação de culturas com dados da NASA (protótipo)

**Desafio NASA Space Apps 2026 — *Field Shift: Adapting Farms with NASA Data*.**
Ferramenta de apoio à decisão que junta **dados de observação da Terra da NASA (NASA POWER)**, o **solo do local**,
as **características das culturas** e as **prioridades do agricultor** para sugerir rotações que protejam o solo
e se adaptem ao clima que está mudando.

Como funciona:
1. Baixa a série **diária** do **NASA POWER** (chuva, temperatura máx./mín.) para um ponto (padrão: Uberlândia-MG).
2. Calcula a evapotranspiração (Hargreaves) e roda um **balanço hídrico tipo "balde"**, inspirado no modelo SARRA usado pelo ZARC.
3. Para **6 culturas** (soja, milho safrinha, sorgo, feijão, crotalária e braquiária), **36 decêndios de semeadura** e
   **6 níveis de retenção de água do solo** (classes AD1–AD6 do ZARC; o produtor pode informar só o tipo: arenoso, médio, argiloso ou muito argiloso), calcula o **ISNA** por fase e o **risco climático**
   (frequência de anos em que a cultura sofreu falta de água numa fase crítica).
4. **Field Shift:** compara o risco entre **1991–2008** e **2009–2024** para mostrar quais janelas ficaram mais arriscadas.
5. **Recomendador de rotação:** a partir do que o agricultor já plantou, do solo, da data de plantio e das prioridades dele,
   ordena as sequências *safra + safrinha/cobertura* por uma nota que junta **risco climático recente**, **saúde do solo** e **diversificação**.

👉 **Para usar:** no Colab, preencha o formulário **"Dados do produtor"** logo abaixo (listas e barrinhas, sem mexer no código) e clique em **Ambiente de execução → Executar tudo**.

⚠️ Durações das fases, Kc, limites de ISNA e notas de saúde do solo são **provisórios**: calibrar com a documentação do ZARC
e com agrônomos antes de qualquer uso real.

In [ ]:
# ===== Dados do produtor =====
#@title 🌱 Dados do produtor (preencha e depois clique em Ambiente de execução → Executar tudo)
#@markdown No Colab esta célula vira um formulário: é só escolher nas listas, sem mexer no código.

MUNICIPIO = "Uberlândia-MG"  #@param {type:"string"}
LAT = -18.9186  #@param {type:"number"}
LON = -48.2772  #@param {type:"number"}

#@markdown **O que você plantou nas duas últimas safras?**
PENULTIMA = "soja"  #@param ["soja", "milho_safrinha", "sorgo", "feijao", "crotalaria", "braquiaria", "nenhuma"]
ULTIMA = "milho_safrinha"  #@param ["soja", "milho_safrinha", "sorgo", "feijao", "crotalaria", "braquiaria", "nenhuma"]

#@markdown **Como é o seu solo?** Se não souber, escolha "não sei". Se tiver análise de solo, pode escolher a classe AD.
SOLO = "argiloso"  #@param ["arenoso", "médio", "argiloso", "muito argiloso", "não sei", "AD1", "AD2", "AD3", "AD4", "AD5", "AD6"]

#@markdown **Quando pretende plantar?**
DATA_PLANTIO = "2026-10-21"  #@param {type:"date"}

#@markdown **O que é mais importante para você?** (0 = nada, 1 = muito)
PESO_RISCO = 0.5  #@param {type:"slider", min:0, max:1, step:0.1}
PESO_SOLO = 0.3  #@param {type:"slider", min:0, max:1, step:0.1}
PESO_DIVERSIDADE = 0.2  #@param {type:"slider", min:0, max:1, step:0.1}

HISTORICO = [c for c in (PENULTIMA, ULTIMA) if c != "nenhuma"]
PRIORIDADES = {
    "risco": PESO_RISCO,              # menos chance de perder a lavoura por falta de chuva
    "solo": PESO_SOLO,                # mais saúde do solo (palha, nitrogênio, raízes)
    "diversidade": PESO_DIVERSIDADE,  # variar as culturas em relação ao histórico
}

# ===== Período dos dados =====
ANO_INI, ANO_FIM = 1991, 2025
PERIODO_1 = range(1991, 2009)   # 1991–2008
PERIODO_2 = range(2009, 2025)   # 2009–2024 (safras com ciclo completo nos dados)

In [ ]:
# ===== Download NASA POWER (diário, comunidade AG) =====
import requests, pandas as pd, numpy as np

url = "https://power.larc.nasa.gov/api/temporal/daily/point"
params = {
    "parameters": "PRECTOTCORR,T2M_MAX,T2M_MIN",
    "community": "AG",
    "latitude": LAT, "longitude": LON,
    "start": f"{ANO_INI}0101", "end": f"{ANO_FIM}1231",
    "format": "JSON",
}
r = requests.get(url, params=params, timeout=120)
r.raise_for_status()
p = r.json()["properties"]["parameter"]

df = pd.DataFrame({
    "chuva": pd.Series(p["PRECTOTCORR"]),
    "tmax": pd.Series(p["T2M_MAX"]),
    "tmin": pd.Series(p["T2M_MIN"]),
})
df.index = pd.to_datetime(df.index, format="%Y%m%d")
df = df.replace(-999, np.nan)
print("Dias faltantes:\n", df.isna().sum())
df = df.interpolate(limit=5).fillna({"chuva": 0})
df.describe()

In [ ]:
# ===== Funções do modelo (balanço hídrico, ISNA e risco) =====
import numpy as np
import pandas as pd

# ---------------- Solo: água disponível por classe (mm por cm de solo) ----------------
# Faixas da IN SPA/MAPA nº 1/2022: AD1 0,34–0,46 | AD2 0,46–0,61 | AD3 0,61–0,80 |
# AD4 0,80–1,06 | AD5 1,06–1,40 | AD6 ≥ 1,40. Usamos um valor dentro de cada faixa.
# O ZARC não dá nome de textura às classes; os nomes abaixo descrevem a retenção de água.
CLASSES_AD = {"AD1": 0.40, "AD2": 0.535, "AD3": 0.705, "AD4": 0.93, "AD5": 1.23, "AD6": 1.50}
NOMES_AD = {
    "AD1": "AD1 – retenção muito baixa",
    "AD2": "AD2 – retenção baixa",
    "AD3": "AD3 – retenção média",
    "AD4": "AD4 – retenção média-alta",
    "AD5": "AD5 – retenção alta",
    "AD6": "AD6 – retenção muito alta",
}
# O produtor normalmente sabe o tipo de solo, não a classe AD. Tradução APROXIMADA (o ZARC define a classe pela
# análise granulométrica — areia, silte e argila —, então o ideal é usar a classe AD da análise de solo):
TEXTURA_PARA_AD = {"arenoso": "AD2", "médio": "AD3", "medio": "AD3", "argiloso": "AD4", "muito argiloso": "AD5",
                   "não sei": "AD3", "nao sei": "AD3"}  # sem informação: usa retenção média


def resolver_solo(solo):
    s = solo.strip()
    if s.upper() in CLASSES_AD:
        return s.upper()
    ad = TEXTURA_PARA_AD.get(s.lower())
    if ad is None:
        raise ValueError(f"Solo '{solo}' não reconhecido. Use {list(TEXTURA_PARA_AD)} ou AD1–AD6.")
    print(f"Solo '{s}' tratado como {NOMES_AD[ad]} (aproximação; confirme com a análise de solo).")
    return ad


SOLO_AD = resolver_solo(SOLO)
PROF_RAIZ_CM = 60  # profundidade efetiva usada no ZARC soja
P_DEPLECAO = 0.5   # fração da CAD consumida sem estresse

# ---------------- Culturas (VALORES PROVISÓRIOS — calibrar com ZARC/agrônomos) ----------------
# fases: (nome, dias, Kc) | limites: ISNA mínimo nas fases críticas
# saude_solo (0–1): palha, raízes e fixação de nitrogênio | papel: grão ou cobertura
CULTURAS = {
    "soja": {"nome": "Soja", "familia": "leguminosa", "papel": "grão", "saude_solo": 0.55,
             "fases": [("F1_inicial", 20, 0.40), ("F2_vegetativo", 30, 0.80),
                       ("F3_reprodutivo", 45, 1.15), ("F4_maturacao", 25, 0.60)],
             "limites": {"F1_inicial": 0.50, "F3_reprodutivo": 0.55}},
    "milho_safrinha": {"nome": "Milho safrinha", "familia": "gramínea", "papel": "grão", "saude_solo": 0.65,
             "fases": [("F1_inicial", 20, 0.40), ("F2_vegetativo", 35, 0.80),
                       ("F3_reprodutivo", 40, 1.20), ("F4_maturacao", 25, 0.60)],
             "limites": {"F1_inicial": 0.50, "F3_reprodutivo": 0.55}},
    "sorgo": {"nome": "Sorgo", "familia": "gramínea", "papel": "grão", "saude_solo": 0.65,
             "fases": [("F1_inicial", 20, 0.40), ("F2_vegetativo", 30, 0.75),
                       ("F3_reprodutivo", 35, 1.05), ("F4_maturacao", 25, 0.60)],
             "limites": {"F1_inicial": 0.45, "F3_reprodutivo": 0.50}},
    "feijao": {"nome": "Feijão", "familia": "leguminosa", "papel": "grão", "saude_solo": 0.45,
             "fases": [("F1_inicial", 15, 0.40), ("F2_vegetativo", 25, 0.75),
                       ("F3_reprodutivo", 30, 1.10), ("F4_maturacao", 20, 0.50)],
             "limites": {"F1_inicial": 0.50, "F3_reprodutivo": 0.60}},
    "crotalaria": {"nome": "Crotalária (cobertura)", "familia": "leguminosa", "papel": "cobertura", "saude_solo": 0.90,
             "fases": [("F1_inicial", 15, 0.40), ("F2_vegetativo", 35, 0.80), ("F3_reprodutivo", 40, 1.00)],
             "limites": {"F1_inicial": 0.45}},
    "braquiaria": {"nome": "Braquiária (cobertura)", "familia": "gramínea", "papel": "cobertura", "saude_solo": 0.90,
             "fases": [("F1_inicial", 20, 0.40), ("F2_vegetativo", 80, 0.85)],
             "limites": {"F1_inicial": 0.40}},
}


def radiacao_extraterrestre(lat_graus, doy):
    """Ra em mm/dia equivalentes (FAO-56)."""
    phi = np.radians(lat_graus)
    dr = 1 + 0.033 * np.cos(2 * np.pi * doy / 365)
    delta = 0.409 * np.sin(2 * np.pi * doy / 365 - 1.39)
    ws = np.arccos(np.clip(-np.tan(phi) * np.tan(delta), -1, 1))
    ra_mj = (24 * 60 / np.pi) * 0.0820 * dr * (
        ws * np.sin(phi) * np.sin(delta) + np.cos(phi) * np.cos(delta) * np.sin(ws))
    return 0.408 * ra_mj


def eto_hargreaves(tmax, tmin, lat, doy):
    tmed = (tmax + tmin) / 2
    ra = radiacao_extraterrestre(lat, doy)
    return np.maximum(0.0023 * (tmed + 17.8) * np.sqrt(np.maximum(tmax - tmin, 0)) * ra, 0)


def _passo_balde(S, chuva, etm, cad):
    """Um dia do balanço hídrico. Funciona com números ou vetores numpy."""
    S = np.minimum(cad, S + chuva)
    limiar = (1 - P_DEPLECAO) * cad
    ks = np.where(S >= limiar, 1.0, S / limiar)
    etr = np.minimum(etm * ks, S)
    return S - etr, etr


def isna_por_fase(chuva, eto, i0, cad, fases, spinup=30, kc_pousio=0.3):
    """Simula, de uma vez, várias semeaduras (vetores i0 e cad do mesmo tamanho).
    Retorna {fase: vetor de ISNA}. Vetorizado: leve em memória e rápido mesmo em computador simples."""
    i0, cad = np.asarray(i0), np.asarray(cad, dtype=float)
    S = 0.5 * cad
    for t in range(-spinup, 0):  # aquecimento com solo em pousio
        S, _ = _passo_balde(S, chuva[i0 + t], kc_pousio * eto[i0 + t], cad)
    out, t = {}, 0
    for nome, dias, kc in fases:
        s_etr = np.zeros_like(cad)
        s_etm = np.zeros_like(cad)
        for _ in range(dias):
            etm = kc * eto[i0 + t]
            S, etr = _passo_balde(S, chuva[i0 + t], etm, cad)
            s_etr += etr
            s_etm += etm
            t += 1
        out[nome] = np.divide(s_etr, s_etm, out=np.ones_like(s_etr), where=s_etm > 0)
    return out


def datas_decendios(ano):
    return [pd.Timestamp(ano, m, d) for m in range(1, 13) for d in (1, 11, 21)]


def simular_risco(df, lat, anos, cultura="soja", spinup=30):
    """df: índice diário (DatetimeIndex) com colunas chuva, tmax, tmin.
    Retorna uma linha por ano × decêndio × classe de solo com o ISNA e se houve falha."""
    cfg = CULTURAS[cultura]
    df = df.sort_index()
    eto = eto_hargreaves(df.tmax.values, df.tmin.values, lat, df.index.day_of_year.values)
    chuva = df.chuva.values
    pos = pd.Series(np.arange(len(df)), index=df.index)
    n_ciclo = sum(d for _, d, _ in cfg["fases"])

    comb = pd.DataFrame([(c, a, k, d) for c in CLASSES_AD for a in anos
                         for k, d in enumerate(datas_decendios(a), start=1)],
                        columns=["classe_solo", "ano", "decendio", "data_semeadura"])
    comb["i0"] = comb.data_semeadura.map(pos)
    comb = comb.dropna(subset=["i0"])
    comb["i0"] = comb.i0.astype(int)
    comb = comb[(comb.i0 - spinup >= 0) & (comb.i0 + n_ciclo <= len(df))].reset_index(drop=True)

    cad = comb.classe_solo.map(CLASSES_AD).values * PROF_RAIZ_CM
    isna = isna_por_fase(chuva, eto, comb.i0.values, cad, cfg["fases"], spinup=spinup)
    for fase, v in isna.items():
        comb[f"isna_{fase}"] = v
    comb["falha"] = np.logical_or.reduce([isna[f] < lim for f, lim in cfg["limites"].items()])
    comb["cultura"] = cultura
    return comb.drop(columns="i0")


def classificar(risco):
    if risco <= 0.20:
        return "20%"
    if risco <= 0.30:
        return "30%"
    if risco <= 0.40:
        return "40%"
    return "não indicado"


def tabua_risco(sim, anos=None):
    s = sim if anos is None else sim[sim.ano.isin(anos)]
    t = s.groupby(["cultura", "classe_solo", "decendio"], as_index=False).agg(
        risco=("falha", "mean"), n_anos=("ano", "nunique"))
    t["faixa"] = t.risco.apply(classificar)
    return t

In [ ]:
# ===== Simulação para todas as culturas e todos os anos =====
sim = pd.concat([simular_risco(df, LAT, range(ANO_INI, ANO_FIM), c) for c in CULTURAS], ignore_index=True)
tabua = tabua_risco(sim)
tabua.assign(classe_solo=tabua.classe_solo.map(NOMES_AD)).to_csv("tabua_risco_nasa_power.csv", index=False)
print(sim.shape)

# Tábua da soja no formato do ZARC: decêndio × nível de retenção de água do solo
(tabua[tabua.cultura == "soja"]
 .assign(classe_solo=lambda t: t.classe_solo.map(NOMES_AD))
 .pivot(index="decendio", columns="classe_solo", values="faixa"))

In [ ]:
# ===== Field Shift: o risco mudou entre os dois períodos? =====
import matplotlib.pyplot as plt

t1 = tabua_risco(sim, PERIODO_1).assign(periodo="1991–2008")
t2 = tabua_risco(sim, PERIODO_2).assign(periodo="2009–2024")
comp = t1.merge(t2, on=["cultura", "classe_solo", "decendio"], suffixes=("_p1", "_p2"))
comp["variacao_pp"] = (comp.risco_p2 - comp.risco_p1) * 100
comp.assign(classe_solo=comp.classe_solo.map(NOMES_AD)).to_csv("comparacao_periodos.csv", index=False)

meses = ["Jan","Fev","Mar","Abr","Mai","Jun","Jul","Ago","Set","Out","Nov","Dez"]
rotulos = [f"{m}-{d}" for m in meses for d in ("1","2","3")]

c = comp[(comp.cultura == "soja") & (comp.classe_solo == SOLO_AD)]
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(c.decendio, c.risco_p1 * 100, label="1991–2008")
ax.plot(c.decendio, c.risco_p2 * 100, label="2009–2024")
for lim in (20, 30, 40):
    ax.axhline(lim, ls=":", lw=0.8, color="gray")
ax.set_xticks(range(1, 37), rotulos, rotation=90, fontsize=7)
ax.set_ylabel("Risco de falha (%)")
ax.set_title(f"Soja (ciclo médio) — {MUNICIPIO} — {NOMES_AD[SOLO_AD]}: risco por decêndio de semeadura")
ax.legend(); plt.tight_layout(); plt.show()

piv = comp[comp.cultura == "soja"].pivot(index="classe_solo", columns="decendio", values="variacao_pp")
fig, ax = plt.subplots(figsize=(12, 3))
im = ax.imshow(piv.values, cmap="RdBu_r", vmin=-30, vmax=30, aspect="auto")
ax.set_yticks(range(len(piv.index)), [NOMES_AD[i] for i in piv.index])
ax.set_xticks(range(36), rotulos, rotation=90, fontsize=7)
fig.colorbar(im, label="Variação do risco (p.p.)")
ax.set_title("Soja: mudança no risco entre períodos (vermelho = ficou mais arriscado)")
plt.tight_layout(); plt.show()

In [ ]:
# ===== Tendência da chuva na estação (out–mar) por safra =====
from scipy.stats import kendalltau

d = df.copy()
d["safra"] = np.where(d.index.month >= 7, d.index.year, d.index.year - 1)
est = d[d.index.month.isin([10, 11, 12, 1, 2, 3])].groupby("safra").chuva.sum()
est = est.loc[ANO_INI:ANO_FIM - 1]
tau, pval = kendalltau(est.index, est.values)
incl = np.polyfit(est.index, est.values, 1)[0]
print(f"Chuva out–mar: tendência {incl:+.1f} mm/ano | Kendall tau = {tau:.2f}, p = {pval:.3f}")
est.plot(marker="o", figsize=(10, 3), title="Chuva acumulada out–mar por safra (NASA POWER)", ylabel="mm");

## Recomendador de rotação

Para cada sequência **safra de grão → segundo cultivo** (safrinha, planta de cobertura ou pousio), a ferramenta calcula:

- **Risco climático recente (2009–2024):** em quantos anos *alguma* das culturas da sequência teria sofrido falta de água,
  plantando na data informada e emendando o segundo cultivo logo após a colheita.
- **Saúde do solo:** palha, raízes e nitrogênio que as culturas deixam, com bônus por alternar **leguminosa ↔ gramínea**
  e por incluir **planta de cobertura**.
- **Diversificação:** quanto a sequência foge do que já foi plantado; repetir a última cultura é descartado.

A **nota final (0–100)** é a média dessas três notas, ponderada pelas **prioridades** do agricultor.
A coluna **mudança** mostra quanto o risco da sequência mudou entre 1991–2008 e 2009–2024 (o *field shift*).

In [ ]:
# ===== Funções do recomendador de rotação =====

def decendio_da_data(data):
    data = pd.Timestamp(data)
    return (data.month - 1) * 3 + min((data.day - 1) // 10, 2) + 1


def proximo_decendio(ano, k, dias):
    """Primeiro decêndio que começa depois de `dias` dias a partir do decêndio k do ano. Retorna (anos a somar, k)."""
    fim = datas_decendios(ano)[k - 1] + pd.Timedelta(days=dias)
    for a in (fim.year, fim.year + 1):
        for j, d in enumerate(datas_decendios(a), start=1):
            if d >= fim:
                return a - ano, j


def _falhas(sim, cultura, solo, k):
    s = sim[(sim.cultura == cultura) & (sim.classe_solo == solo) & (sim.decendio == k)]
    return s.set_index("ano").falha


def recomendar_rotacao(sim, historico, solo, data_plantio, prioridades, top=8):
    k1 = decendio_da_data(data_plantio)
    ultima = historico[-1] if historico else None
    fam_ultima = CULTURAS[ultima]["familia"] if ultima in CULTURAS else None
    pesos = {k: prioridades.get(k, 0) for k in ("risco", "solo", "diversidade")}
    soma = sum(pesos.values()) or 1

    linhas = []
    for a, ca in CULTURAS.items():
        if a == ultima or ca["papel"] != "grão":
            continue  # safra principal é de grão (renda) e não repete a última cultura
        f1 = _falhas(sim, a, solo, k1)
        n_a = sum(d for _, d, _ in ca["fases"])
        ano_ref = f1.index.min() if len(f1) else ANO_INI
        da, k2 = proximo_decendio(ano_ref, k1, n_a)
        for b in [None] + [c for c in CULTURAS if c != a]:
            if b is None:
                f = f1
            else:
                f2 = _falhas(sim, b, solo, k2)
                f2.index = f2.index - da          # alinha o 2º cultivo ao ano da safra principal
                f = pd.concat([f1, f2], axis=1, join="inner").any(axis=1)  # falha se qualquer uma falhar
            r_rec = f[f.index.isin(PERIODO_2)].mean()
            r_ant = f[f.index.isin(PERIODO_1)].mean()
            if np.isnan(r_rec):
                continue

            seq = [a] + ([b] if b else [])
            saude = np.mean([CULTURAS[c]["saude_solo"] for c in seq]) * (1 if b else 0.6)  # pousio deixa o solo descoberto
            if fam_ultima and CULTURAS[a]["familia"] != fam_ultima:
                saude += 0.10
            if b and CULTURAS[a]["familia"] != CULTURAS[b]["familia"]:
                saude += 0.10
            if any(CULTURAS[c]["papel"] == "cobertura" for c in seq):
                saude += 0.05
            saude = min(saude, 1.0)
            divers = np.mean([c not in historico for c in seq])

            nota = 100 * (pesos["risco"] * (1 - r_rec) + pesos["solo"] * saude + pesos["diversidade"] * divers) / soma

            motivos = [f"risco {r_rec:.0%} nos últimos anos"]
            if fam_ultima and CULTURAS[a]["familia"] != fam_ultima:
                motivos.append(f"alterna com {CULTURAS[ultima]['nome'].lower()}")
            if any(CULTURAS[c]["familia"] == "leguminosa" for c in seq):
                motivos.append("leguminosa fixa nitrogênio")
            if any(CULTURAS[c]["papel"] == "cobertura" for c in seq):
                motivos.append("cobertura protege o solo")
            if b is None:
                motivos.append("solo fica descoberto depois")

            linhas.append({
                "safra": ca["nome"],
                "depois": CULTURAS[b]["nome"] if b else "Pousio",
                "risco": r_rec,
                "mudanca_pp": (r_rec - r_ant) * 100,
                "saude_solo": saude,
                "diversidade": divers,
                "nota": nota,
                "motivo": "; ".join(motivos),
            })
    rec = pd.DataFrame(linhas).sort_values("nota", ascending=False).head(top).reset_index(drop=True)
    rec.index += 1
    return rec, k1, k2

In [ ]:
# ===== Painel do produtor =====
rec, k1, k2 = recomendar_rotacao(sim, HISTORICO, SOLO_AD, DATA_PLANTIO, PRIORIDADES)
melhor = rec.iloc[0]

print(f"📍 {MUNICIPIO} | solo: {NOMES_AD[SOLO_AD]} | plantio: {rotulos[k1 - 1]} (decêndio {k1})")
print(f"🌱 Histórico: {' → '.join(CULTURAS[c]['nome'] for c in HISTORICO)}\n")
print(f"✅ Sugestão: {melhor.safra} e depois {melhor.depois} (nota {melhor.nota:.0f}/100)")
print(f"   Por quê: {melhor.motivo}.")
tend = "subiu" if melhor.mudanca_pp > 0 else "caiu"
print(f"   O risco dessa sequência {tend} {abs(melhor.mudanca_pp):.0f} p.p. entre 1991–2008 e 2009–2024.")

tabela = rec.assign(risco=(rec.risco * 100).round(0).astype(int).astype(str) + "%",
                    mudanca_pp=rec.mudanca_pp.round(0).map("{:+.0f} p.p.".format),
                    saude_solo=(rec.saude_solo * 100).round(0).astype(int),
                    diversidade=(rec.diversidade * 100).round(0).astype(int),
                    nota=rec.nota.round(0).astype(int))
display(tabela.rename(columns={"safra": "Safra", "depois": "Depois", "risco": "Risco recente",
                               "mudanca_pp": "Mudança", "saude_solo": "Saúde do solo",
                               "diversidade": "Diversidade", "nota": "Nota", "motivo": "Motivo"}))

# Gráfico: nota de cada rotação, separada nas três partes
w = {k: PRIORIDADES.get(k, 0) for k in ("risco", "solo", "diversidade")}
s = sum(w.values()) or 1
partes = pd.DataFrame({
    "Clima (menos risco)": 100 * w["risco"] * (1 - rec.risco) / s,
    "Saúde do solo": 100 * w["solo"] * rec.saude_solo / s,
    "Diversificação": 100 * w["diversidade"] * rec.diversidade / s,
})
partes.index = rec.safra + " → " + rec.depois
ax = partes.iloc[::-1].plot.barh(stacked=True, figsize=(10, 0.5 * len(partes) + 1.5),
                                 color=["#2a78b5", "#5a9a3a", "#d08c2c"])
ax.set_xlabel("Nota (0–100)")
ax.set_title(f"Rotações sugeridas — plantio em {rotulos[k1 - 1]}, {NOMES_AD[SOLO_AD]}")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout(); plt.show()

## Próximos passos
- **Calibrar** fases, Kc e limites de ISNA de cada cultura com a documentação do ZARC; revisar as notas de saúde do solo com agrônomos.
- **Validar:** comparar `tabua_risco_nasa_power.csv` com a tábua oficial do ZARC para o município (concordância das faixas por decêndio).
- **Solo local:** puxar a classe AD de mapas de solo (ex.: SoilGrids/Embrapa) a partir da coordenada, em vez de digitar.
- **Escalar:** rodar para vários municípios (lista de lat/lon) e montar um mapa.
- **Interface:** transformar o painel do produtor numa página simples (formulário → recomendação).